# Single-DQD phase accumulation

Interactive notebook for the `single_dqd_phase_accumulation.py` draft simulation. The model uses the basis

`|up,R>`, `|up,L>`, `|down,R>`, `|down,L>`

and tracks the dynamic qubit phase accumulated by a detuning excursion relative to staying at `epsilon = 0`:

$$\phi_{detuning}(t) = \int_0^t [\omega_{ge}(\epsilon(t')) - \omega_{ge}(0)]\,dt'.$$

Use the widget cell near the bottom to sweep tunnel coupling, field gradient, detuning target, ramp time, and hold time.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display

notebook_dir = Path.cwd()
if not (notebook_dir / "single_dqd_phase_accumulation.py").exists():
    notebook_dir = Path("python/notebooks").resolve()

python_dir = notebook_dir.parent
for path in (notebook_dir, python_dir):
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

from single_dqd_phase_accumulation import (  # noqa: E402
    DQDParams,
    PulseParams,
    make_epsilon_schedule,
    print_summary,
    qubit_splitting,
    run_simulation,
    rx_duration,
)

try:
    import ipywidgets as widgets
    from ipywidgets import interact_manual
except ImportError:
    widgets = None
    interact_manual = None

## Baseline run

Run the default pulse once, then inspect the schedule, phase trace, and final projected qubit state.

In [ ]:
params = DQDParams()
pulse = PulseParams()
result = run_simulation(params, pulse)
print_summary(result)

In [ ]:
def plot_result(result: dict, *, title: str | None = None) -> plt.Figure:
    t = result["t_eval"]
    states = result["states"]
    phase = result["phase"]
    final = result["final"]

    fig, axes = plt.subplots(4, 1, figsize=(9, 9), sharex=True, constrained_layout=True)
    if title:
        fig.suptitle(title)

    axes[0].plot(t, result["epsilon"], color="C0")
    axes[0].set_ylabel("epsilon (ueV)")

    axes[1].plot(t, result["delta_omega"], color="C1")
    axes[1].set_ylabel("delta omega (rad/ns)")

    axes[2].plot(t, phase, color="C2", label="unwrapped")
    axes[2].plot(t, np.angle(np.exp(1j * phase)), color="C3", ls="--", label="mod 2pi")
    axes[2].set_ylabel("phase (rad)")
    axes[2].legend(loc="best")

    basis_pops = np.abs(states) ** 2
    labels = ["|up,R>", "|up,L>", "|down,R>", "|down,L>"]
    for idx, label in enumerate(labels):
        axes[3].plot(t, basis_pops[idx], label=label)
    axes[3].set_ylabel("basis population")
    axes[3].set_xlabel("time (ns)")
    axes[3].legend(loc="best", ncol=2)

    text = (
        f"final Pg={final['pop_g']:.6f}, Pe={final['pop_e']:.6f}, "
        f"leakage={final['leakage']:.2e}, "
        f"phase={phase[-1]:.6f} rad ({np.degrees(phase[-1]):.3f} deg)"
    )
    axes[3].text(0.01, -0.35, text, transform=axes[3].transAxes)
    return fig


plot_result(result, title="Default single-DQD phase accumulation");

## Sweep detuning hold time

This cell avoids full Schrodinger evolution and only integrates the adiabatic phase along each detuning schedule, so it is quick enough for exploratory sweeps.

In [ ]:
def phase_for_hold_time(params: DQDParams, pulse: PulseParams, hold_time: float) -> float:
    local_pulse = PulseParams(
        rx_angle=pulse.rx_angle,
        epsilon_target=pulse.epsilon_target,
        ramp_time=pulse.ramp_time,
        hold_time=hold_time,
        points_per_carrier_cycle=pulse.points_per_carrier_cycle,
        max_step_fraction_of_period=pulse.max_step_fraction_of_period,
    )
    t_rx = rx_duration(params, local_pulse.rx_angle)
    total_time = t_rx + 2 * local_pulse.ramp_time + local_pulse.hold_time
    t = np.linspace(0.0, total_time, 1200)
    eps = make_epsilon_schedule(t_rx, local_pulse)
    omega0 = qubit_splitting(params, epsilon=0.0)
    delta = np.array([qubit_splitting(params, eps(float(ti))) - omega0 for ti in t])
    return float(np.trapezoid(delta, t))


hold_times = np.linspace(0.0, 100.0, 101)
phases = np.array([phase_for_hold_time(params, pulse, ht) for ht in hold_times])

fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
ax.plot(hold_times, phases, label="unwrapped")
ax.plot(hold_times, np.angle(np.exp(1j * phases)), label="mod 2pi", ls="--")
ax.set_xlabel("hold time (ns)")
ax.set_ylabel("detuning phase (rad)")
ax.legend();

## Interactive controls

The manual button keeps expensive ODE solves from firing on every slider movement. Enable `phase_only` for faster scans; disable it when you want the full state evolution and final populations.

In [ ]:
def run_interactive(
    tc: float = 40.0,
    dBx: float = 2.0,
    Bz: float = 20.0,
    Vac0: float = 1.0,
    epsilon_target: float = 2000.0,
    ramp_time: float = 5.0,
    hold_time: float = 20.0,
    points_per_carrier_cycle: int = 24,
    drive_phase_deg: float = 0.0,
    phase_only: bool = False,
) -> None:
    params = DQDParams(tc=tc, dBx=dBx, Bz=Bz, Vac0=Vac0)
    pulse = PulseParams(
        epsilon_target=epsilon_target,
        ramp_time=ramp_time,
        hold_time=hold_time,
        points_per_carrier_cycle=points_per_carrier_cycle,
    )

    if phase_only:
        phase = phase_for_hold_time(params, pulse, hold_time)
        t_rx = rx_duration(params, pulse.rx_angle)
        total_time = t_rx + 2 * ramp_time + hold_time
        t = np.linspace(0.0, total_time, 1200)
        eps = make_epsilon_schedule(t_rx, pulse)
        epsilon = np.array([eps(float(ti)) for ti in t])
        omega0 = qubit_splitting(params, epsilon=0.0)
        delta = np.array([qubit_splitting(params, eps(float(ti))) - omega0 for ti in t])
        phase_trace = np.concatenate([[0.0], np.cumsum((delta[1:] + delta[:-1]) / 2 * np.diff(t))])

        fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True, constrained_layout=True)
        axes[0].plot(t, epsilon)
        axes[0].set_ylabel("epsilon (ueV)")
        axes[1].plot(t, delta)
        axes[1].set_ylabel("delta omega (rad/ns)")
        axes[2].plot(t, phase_trace, label="unwrapped")
        axes[2].plot(t, np.angle(np.exp(1j * phase_trace)), ls="--", label="mod 2pi")
        axes[2].set_ylabel("phase (rad)")
        axes[2].set_xlabel("time (ns)")
        axes[2].legend(loc="best")
        print(f"phase={phase:.9f} rad ({np.degrees(phase):.6f} deg)")
        print(f"phase mod 2pi={np.angle(np.exp(1j * phase)):.9f} rad")
        return

    result = run_simulation(params, pulse, drive_phase=np.deg2rad(drive_phase_deg))
    print_summary(result)
    plot_result(result, title="Interactive single-DQD phase accumulation")


if widgets is None:
    display(HTML("<b>ipywidgets is not installed.</b> Run the cells above directly, or install ipywidgets to use sliders."))
else:
    interact_manual(
        run_interactive,
        tc=widgets.FloatSlider(value=40.0, min=10.0, max=80.0, step=1.0, description="tc"),
        dBx=widgets.FloatSlider(value=2.0, min=0.1, max=10.0, step=0.1, description="dBx"),
        Bz=widgets.FloatSlider(value=20.0, min=1.0, max=60.0, step=1.0, description="Bz"),
        Vac0=widgets.FloatSlider(value=1.0, min=0.05, max=5.0, step=0.05, description="Vac0"),
        epsilon_target=widgets.FloatSlider(value=2000.0, min=0.0, max=5000.0, step=50.0, description="eps target"),
        ramp_time=widgets.FloatSlider(value=5.0, min=0.1, max=50.0, step=0.5, description="ramp ns"),
        hold_time=widgets.FloatSlider(value=20.0, min=0.0, max=200.0, step=1.0, description="hold ns"),
        points_per_carrier_cycle=widgets.IntSlider(value=24, min=8, max=64, step=2, description="points/cycle"),
        drive_phase_deg=widgets.FloatSlider(value=0.0, min=-180.0, max=180.0, step=5.0, description="drive deg"),
        phase_only=widgets.Checkbox(value=False, description="phase only"),
    );